<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB11_Neural_Network_Fundamentals_First_PyTorch_Model_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB11 · Clase 11 — Fundamentos de redes neuronales: tu primer modelo en PyTorch**

## Bloque 3: IA — Deep Learning (apertura)

`NB01`–`NB10` cubrieron el Machine Learning clásico: modelos que trabajan directamente sobre características numéricas diseñadas a mano. Esta clase abre el **Bloque 3 — Deep Learning**: redes neuronales, que aprenden sus propias representaciones internas en vez de depender por completo de características que elegimos a mano. Construimos la comprensión desde la unidad más pequeña posible (un único perceptrón) hasta una red multicapa real y entrenada — usando **[PyTorch](https://pytorch.org/)**, un framework de deep learning muy usado en el que nos apoyaremos durante todo este bloque.

Para que la comparación sea honesta y concreta, entrenamos nuestra primera red sobre un dataset que ya conocemos bien: el dataset real **Sonar (minas vs. rocas)** de `NB08`, y comparamos nuestra red neuronal directamente con los resultados clásicos de `NB08` (árbol de decisión, Random Forest, AdaBoost, SVM).

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar qué calcula un perceptrón, y por qué un único perceptrón solo puede aprender patrones linealmente separables.
- Explicar cómo apilar capas (un Perceptrón Multicapa) y las funciones de activación no lineales superan esa limitación.
- Explicar, a nivel conceptual, cómo aprende una red: forward pass, función de pérdida, backpropagation, descenso de gradiente.
- Construir una pequeña red neuronal feedforward en PyTorch (`nn.Module`, un bucle de entrenamiento con `loss.backward()`/`optimizer.step()`).
- Entrenar la red, representar su curva de pérdida, y evaluarla exactamente igual que cualquier otro clasificador de `NB08`.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso del Bloque 2, hoja de ruta del Bloque 3 | 5 min | Teoría |
| 2 | ¿Por qué Deep Learning? Dónde el ML clásico toca techo | 10 min | Teoría |
| 3 | El perceptrón: la unidad básica | 15 min | Teoría |
| 4 | Del perceptrón al Perceptrón Multicapa: capas y funciones de activación | 15 min | Teoría |
| 5 | Cómo aprende una red: pérdida, backpropagation, descenso de gradiente | 15 min | Teoría |
| 6 | Fundamentos de PyTorch: tensores y carga de nuestro dataset real | 15 min | Práctica |
| 7 | Práctica: construir y entrenar un primer clasificador MLP | 20 min | Práctica |
| 8 | Evaluar nuestra red y compararla con los modelos clásicos de NB08 | 15 min | Práctica |
| 9 | Una primera mirada al sobreajuste en redes neuronales | 5 min | Teoría |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una guía aproximada, no un guion cerrado.

---

## 1. Repaso: dónde estamos

- **Bloque 1** (`NB01`): historia y contexto de la IA.
- **Bloque 2** (`NB02`–`NB10`): el conjunto completo de herramientas de ML clásico — Python/NumPy/Pandas, el flujo supervisado, árboles/ensembles/SVM, aprendizaje no supervisado, y un proyecto completo y ajustado.
- **Bloque 3** (empieza hoy): Deep Learning — redes neuronales, construidas desde los primeros principios y entrenadas de verdad.

Este es el bloque que cubre la brecha más grande identificada al comparar este curso con su guía docente oficial: el Deep Learning no tenía ningún contenido práctico antes de este rediseño.

---

## 2. ¿Por qué Deep Learning? Dónde el ML clásico toca techo

Todos los modelos del Bloque 2 trabajaban sobre **características tabulares, diseñadas a mano**: `distance`, `engine_efficiency`, las 60 bandas de frecuencia del sonar, los coeficientes de geometría del casco. Una persona (o un pipeline de datos) decidía de antemano qué números importaban.

Eso funciona bien cuando:
- Los datos ya son naturalmente tabulares (hojas de cálculo, registros de sensores, datos estructurados).
- Una persona experta en el dominio puede nombrar las características relevantes (p. ej., la ingeniería naval ya sabe que el número de Froude importa para la resistencia).

Funciona peor cuando:
- Los datos en bruto son **no estructurados** — una imagen, una forma de onda de audio, texto sin procesar, una secuencia larga de sensores — donde "las características correctas" no son obvias, o hay demasiadas posibles como para diseñarlas a mano.
- Los patrones que importan son **jerárquicos**: un borde → una forma → un objeto, en una imagen; un fonema → una palabra → una frase, en audio/texto.

Las **redes neuronales** resuelven esto `aprendiendo sus propias representaciones intermedias directamente a partir de datos más en bruto, capa a capa`, en vez de exigirnos diseñar a mano cada característica. Esto es *precisamente* lo que motiva las próximas clases: `NB13` (CNN) aplicará esto directamente a imágenes reales de inspección submarina, donde diseñar a mano "las características correctas" sería mucho más difícil que en nuestros datasets tabulares de sonar/buques.

Hoy, sin embargo, empezamos deliberadamente con **datos tabulares familiares** (Sonar, el mismo que en `NB08`) para que la *única* variable nueva sea el propio modelo — todo lo demás (los datos, la evaluación) sigue siendo comparable a lo que ya conoces.

---

## 3. El perceptrón: la unidad básica

Un **[perceptrón](https://en.wikipedia.org/wiki/Perceptron)** (Rosenblatt, 1958) es la red neuronal más simple posible: toma varias entradas numéricas, multiplica cada una por un **peso** aprendido, suma un **sesgo** (bias), y pasa el resultado por una **función de activación**:

$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b \qquad \hat{y} = f(z)
$$

Vamos a plasmar esa estructura en un dibujo — un único perceptrón con 4 entradas de ejemplo, un término de sesgo, y una salida:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 5))

input_labels = ["x1", "x2", "x3", "x4"]
input_y = [4, 3, 2, 1]
input_x = 0
neuron_x, neuron_y = 3.5, 2.5

# Input nodes
for label, y in zip(input_labels, input_y):
    ax.add_patch(patches.Circle((input_x, y), 0.3, facecolor="lightblue", edgecolor="black", zorder=3))
    ax.text(input_x, y, label, ha="center", va="center", fontsize=11, zorder=4)

# Bias node (drawn separately, feeds in like an extra input)
bias_y = 0
ax.add_patch(patches.Circle((input_x, bias_y), 0.3, facecolor="khaki", edgecolor="black", zorder=3))
ax.text(input_x, bias_y, "b", ha="center", va="center", fontsize=11, zorder=4)

# Neuron: weighted sum + activation
ax.add_patch(patches.Circle((neuron_x, neuron_y), 0.55, facecolor="lightcoral", edgecolor="black", zorder=3))
ax.text(neuron_x, neuron_y, "Σ  f", ha="center", va="center", fontsize=12, zorder=4)

# Arrows from inputs to the neuron, labeled with weights
for label, y, w in zip(input_labels, input_y, ["w1", "w2", "w3", "w4"]):
    ax.annotate("", xy=(neuron_x - 0.55, neuron_y), xytext=(input_x + 0.3, y),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text((input_x + neuron_x) / 2 - 0.3, (y + neuron_y) / 2, w, fontsize=10, color="darkblue")

# Bias arrow (dashed, to distinguish it from the real inputs)
ax.annotate("", xy=(neuron_x - 0.55, neuron_y), xytext=(input_x + 0.3, bias_y),
            arrowprops=dict(arrowstyle="->", lw=1.5, linestyle="dashed", color="darkgoldenrod"))

# Output arrow
output_x = 6.5
ax.annotate("", xy=(output_x, neuron_y), xytext=(neuron_x + 0.55, neuron_y),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.text(output_x + 0.2, neuron_y, "y_hat", fontsize=13, va="center")

ax.set_xlim(-1, 8)
ax.set_ylim(-1, 5)
ax.axis("off")
ax.set_title("Structure of a single perceptron")
plt.show()

Cada entrada $x_i$ se multiplica por su propio peso $w_i$ (las flechas continuas); el sesgo $b$ (flecha discontinua) desplaza el resultado independientemente de cualquier entrada; la neurona suma todo ($\Sigma$) y aplica la función de activación $f$ para producir la salida $\hat{y}$. **Los pesos y el sesgo son exactamente lo que aprende el entrenamiento** — la *forma* del diagrama (cuántas entradas, cómo se conectan las flechas) la fija la arquitectura que elegimos; los *números sobre las flechas* son lo que ajusta el descenso de gradiente.

Si $f$ es la función logística/sigmoide, esto es exactamente la **Regresión Logística** de `NB07` disfrazada (con una función escalón en su lugar, es el perceptrón original de Rosenblatt de 1958, entrenado con una regla distinta al descenso de gradiente) — un ancla útil: un único perceptrón no es una idea nueva, `es la forma que tiene una red neuronal de expresar un modelo que ya entiendes`.

**La trampa**: un único perceptrón solo puede separar datos con una línea recta (o un hiperplano) — solo puede aprender patrones **linealmente separables**. Muchos problemas reales (probablemente incluidos nuestros datos de sonar — recuerda que el SVM necesitó un kernel RBF no lineal en `NB08` para funcionar bien) no son linealmente separables. Esa limitación es precisamente lo que motiva la siguiente sección.

> **Para saber más**: [Perceptrón (Wikipedia)](https://en.wikipedia.org/wiki/Perceptron).

---

## 4. Del perceptrón al Perceptrón Multicapa

Apila perceptrones en **capas**, alimentando la salida de cada capa como entrada de la siguiente, y obtienes un **Perceptrón Multicapa (MLP)**: una capa de entrada, una o más **capas ocultas**, y una capa de salida.

Un detalle importa enormemente: si cada capa solo calcula una suma ponderada (sin no linealidad), apilar capas no tiene sentido matemáticamente — `cualquier cadena de funciones lineales se colapsa de nuevo en una única función lineal`, no más potente que un solo perceptrón. Las **funciones de activación no lineales** entre capas son lo que realmente le da poder a la profundidad.

| Activación | Fórmula | Uso típico |
|---|---|---|
| **Sigmoide** | $\dfrac{1}{1+e^{-z}}$ | Capa de salida para clasificación binaria (comprime a 0–1, como la Regresión Logística de `NB07`) |
| **ReLU** (Rectified Linear Unit) | $\max(0, z)$ | La opción por defecto para capas ocultas — rápida, simple, evita un problema de entrenamiento que la sigmoide tiene en redes profundas |
| **Tanh** | $\dfrac{e^z - e^{-z}}{e^z + e^{-z}}$ | Similar a la sigmoide, pero centrada en 0 |

Usaremos **ReLU** en nuestras capas ocultas (práctica habitual) y una salida **equivalente a sigmoide** para nuestra predicción binaria mina/roca.

> **Para saber más**: [Perceptrón multicapa (Wikipedia)](https://en.wikipedia.org/wiki/Multilayer_perceptron) · [Función de activación (Wikipedia)](https://en.wikipedia.org/wiki/Activation_function).

Ver las tres curvas juntas hace concretas sus diferencias — dónde satura (se aplana) cada una y dónde se mantiene lineal:

In [ ]:
import numpy as np

z = np.linspace(-5, 5, 200)
sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)
tanh_curve = np.tanh(z)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(z, sigmoid, label="Sigmoid")
ax.plot(z, relu, label="ReLU")
ax.plot(z, tanh_curve, label="Tanh")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("z (input to the activation)")
ax.set_ylabel("f(z)")
ax.set_title("Common activation functions")
ax.legend()
plt.show()


---

## 5. Cómo aprende una red

Entrenar una red significa encontrar pesos y sesgos que hagan que sus predicciones se ajusten a la realidad lo mejor posible. Tres ingredientes:

1. Una **función de pérdida** mide cuánto se equivoca una predicción. Para clasificación binaria (nuestra tarea del sonar), usamos **entropía cruzada binaria** — conceptualmente similar a las métricas de clasificación de `NB07`, pero diferenciable, lo cual importa para el paso 3.
2. **Backpropagation** calcula, para cada peso de la red, *cuánto contribuyó ese peso concreto al error actual* — de forma eficiente, aplicando la regla de la cadena del cálculo hacia atrás a través de la red, capa por capa.
3. El **descenso de gradiente** empuja entonces cada peso un poco en la dirección que reduce la pérdida, se repite durante muchos pasos pequeños (**épocas**), y — si todo va bien — la pérdida baja y las predicciones mejoran.

$$
w \leftarrow w - \eta \frac{\partial \text{Loss}}{\partial w}
$$

donde $\eta$ (la **tasa de aprendizaje**) controla el tamaño de cada paso: `demasiado grande y el entrenamiento puede divergir; demasiado pequeña y el entrenamiento avanza a paso de tortuga`. En la práctica, no calculamos nada de esto a mano — `loss.backward()` de PyTorch se encarga de la backpropagation automáticamente, y un **optimizador** (usaremos **Adam**, una versión refinada del descenso de gradiente) se encarga de actualizar los pesos.

> **Para saber más**: [Backpropagation (Wikipedia)](https://en.wikipedia.org/wiki/Backpropagation) · [Descenso de gradiente (Wikipedia)](https://en.wikipedia.org/wiki/Gradient_descent) · [Entropía cruzada (Wikipedia)](https://en.wikipedia.org/wiki/Cross-entropy).

---

## 6. Fundamentos de PyTorch: tensores y nuestro dataset real

La estructura de datos central de PyTorch es el **tensor** — como un array de NumPy (`NB02`), pero con dos capacidades extra que necesitamos para deep learning: puede rastrear gradientes automáticamente (para la backpropagation), y puede ejecutarse en GPU. Colab viene con PyTorch preinstalado, así que podemos importarlo directamente.

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

**Pruébalo tú mismo**: antes de trabajar con el dataset real, familiarízate con las operaciones básicas de tensores — crea un tensor pequeño, comprueba su forma, calcula una reducción, y convierte hacia/desde NumPy (las mismas operaciones de array de `NB02`, con el rastreo de gradientes añadido).

In [ ]:
sample_tensor = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print("Shape:", sample_tensor.shape)
print("Mean:", sample_tensor.mean().item())
print("As NumPy array:\n", sample_tensor.numpy())

back_to_tensor = torch.from_numpy(sample_tensor.numpy() * 2)
print("\nDoubled, converted back to a tensor:\n", back_to_tensor)


Vuelve a cargar el dataset real de Sonar — el mismo fichero, las mismas columnas, la misma tarea que en `NB08`:

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X = sonar.drop(columns="label").values
y = (sonar["label"] == "M").astype(int).values
print(X.shape, y.shape)

Divide y escala exactamente igual que en `NB08` — las redes neuronales son, si acaso, *más* sensibles a entradas sin escalar de lo que era el SVM, ya que valores de entrada grandes pueden hacer inestable el entrenamiento:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Convierte los arrays de NumPy en tensores de PyTorch — el formato que espera cualquier modelo de PyTorch:

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

X_train_t.shape, y_train_t.shape

---

## 7. Práctica: construir y entrenar un primer clasificador MLP

Define la red: 60 entradas (nuestras bandas de frecuencia) → una capa oculta de 32 unidades → una capa oculta de 16 unidades → 1 salida (probabilidad de mina). Es una arquitectura pequeña y deliberadamente simple — un punto de partida, no un modelo final ajustado.

In [ ]:
class SonarMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

model = SonarMLP(n_features=X_train_t.shape[1])
model

**Pruébalo tú mismo**: evalúa este modelo recién creado exactamente como harás en la Sección 8 — sus pesos todavía son aleatorios, nunca se han ajustado con descenso de gradiente. ¿Qué precisión predirías, y coincide el número real con tu estimación?

In [ ]:
model.eval()
with torch.no_grad():
    untrained_logits = model(X_test_t)
    untrained_preds = (torch.sigmoid(untrained_logits) > 0.5).float()

untrained_accuracy = (untrained_preds == y_test_t).float().mean().item()
print("Untrained (random weights) test accuracy:", round(untrained_accuracy, 3))

model.train()   # switch back before training


Aproximadamente al nivel del azar, como se esperaba — la red todavía no ha aprendido nada. Guarda este número en mente para la Sección 8, donde lo compararemos con la precisión real del modelo entrenado.

Obtenemos una puntuación en bruto ("logit") en vez de una probabilidad 0–1 directamente — `BCEWithLogitsLoss` aplica la sigmoide y calcula la pérdida a la vez, en un único paso más estable numéricamente. Define la función de pérdida y el optimizador:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Ahora el bucle de entrenamiento — el corazón del deep learning, y merece la pena leerlo línea a línea la primera vez: en cada época, calcula las predicciones (forward pass), calcula la pérdida, calcula los gradientes (backward pass), y actualiza los pesos.

In [ ]:
n_epochs = 200
train_losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{n_epochs} - training loss: {loss.item():.4f}")

Representa la curva de pérdida — debería bajar de forma constante a medida que avanza el entrenamiento:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss (binary cross-entropy)")
plt.title("MLP training loss over epochs")
plt.show()

**Lee tu propia curva**: una pérdida que desciende suavemente y luego se aplana es exactamente lo que queremos — la red está aprendiendo, y después convergiendo. Una pérdida que salta erráticamente suele significar que la tasa de aprendizaje es demasiado alta; una pérdida que apenas se mueve suele significar que es demasiado baja (o que la red es demasiado pequeña para el patrón).

---

## 8. Evaluar nuestra red y compararla con `NB08`

Cambia el modelo a modo de evaluación (desactiva comportamientos exclusivos del entrenamiento que veremos en `NB12`) y predice sobre el conjunto de test reservado, exactamente como haríamos con cualquier clasificador de `NB08`:

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs > 0.5).float()

accuracy = (test_preds == y_test_t).float().mean().item()
print("Test accuracy:", round(accuracy, 3))

Obtén la imagen completa con las mismas herramientas que en `NB08`:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test_t.numpy().ravel()

print(confusion_matrix(y_test_np, y_pred_np))
print()
print(classification_report(y_test_np, y_pred_np, target_names=["Rock", "Mine"]))

Una matriz de confusión visual, igual que se mostró en `NB08` — merece la pena compararla directamente con la referencia sin entrenar de la Sección 7:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(confusion_matrix(y_test_np, y_pred_np), display_labels=["Rock", "Mine"])
disp.plot(cmap="Blues")
plt.title("MLP -- confusion matrix")
plt.show()


**Compara con `NB08`**: ¿cómo se compara la precisión de este MLP sin ajustar con el boxplot de árbol de decisión / Random Forest / AdaBoost / SVM de la Parte 7 de `NB08`? Es habitual que una primera red neuronal sin ajustar quede en algún punto intermedio, no automáticamente por delante — `las redes neuronales generalmente necesitan más datos y un ajuste más cuidadoso que los métodos basados en árboles para mostrar una ventaja real`, que es exactamente para lo que existe `NB12`.

---

## 9. Una primera mirada al sobreajuste en redes neuronales

Recuerda el experimento de profundidad del árbol de decisión de `NB08`: más capacidad no es gratis. Las redes neuronales tienen el mismo modo de fallo, en un mando distinto: más capas, más unidades por capa, y más épocas de entrenamiento `aumentan la capacidad de la red para memorizar sus datos de entrenamiento en vez de aprender el patrón subyacente`.

Con solo 208 ejemplos de sonar y una red 60→32→16→1 (miles de pesos entrenables), ya estamos en un régimen donde el sobreajuste es un riesgo real si el entrenamiento se prolonga lo suficiente — merece la pena vigilar la brecha entre la pérdida de entrenamiento y el rendimiento en test. `NB12` cubre esto en profundidad: curvas de validación, dropout, y early stopping, los equivalentes en redes neuronales del límite `max_depth` de `NB08`.

---

## Resumen de la clase

- Un perceptrón es una suma ponderada más una función de activación — con activación sigmoide, matemáticamente el mismo modelo que la Regresión Logística de `NB07`.
- Apilar capas solo añade poder si hay funciones de activación no lineales (ReLU, sigmoide, tanh) entre ellas.
- Entrenamiento = forward pass (predecir) → pérdida (¿cuánto se equivoca?) → backpropagation (¿de quién es la culpa?) → descenso de gradiente (ajustar los pesos para mejorar).
- Construimos, entrenamos y evaluamos un MLP real de PyTorch sobre el mismo dataset de Sonar y la misma partición train/test que `NB08`, haciendo que la comparación con el ML clásico sea directa y justa.
- Una primera red neuronal sin ajustar no vence automáticamente a modelos clásicos bien ajustados — la profundidad y la flexibilidad hay que ganárselas con una práctica de entrenamiento adecuada, que es lo siguiente.

## Para la próxima clase (NB12)

Entrenaremos redes profundas correctamente: curvas de validación durante el entrenamiento, técnicas de regularización (dropout, early stopping) para combatir el sobreajuste, y un vistazo a distintos optimizadores — convirtiendo la red de hoy, que "funciona", en una en la que realmente confiarías.

## Tarea / Ideas de práctica

1. Cambia el tamaño de las capas ocultas (p. ej., `128` y `64` en vez de `32` y `16`) — ¿mejora la precisión en test, empeora, o apenas cambia?
2. Cambia `n_epochs` a 500 — ¿sigue bajando la pérdida de entrenamiento? ¿Mejora la precisión en test junto con ella, o se estanca (o empeora) mientras la pérdida de entrenamiento sigue bajando?
3. Sustituye `nn.ReLU()` por `nn.Tanh()` en las capas ocultas — ¿se comporta el entrenamiento de forma distinta (revisa la forma de la curva de pérdida)?
4. Prueba una tasa de aprendizaje distinta (`lr=0.01` y `lr=0.0001`) — relaciona lo que observas con la explicación de la Parte 5 sobre qué controla la tasa de aprendizaje.
5. Con tus propias palabras, explica por qué hemos usado `BCEWithLogitsLoss` (entropía cruzada binaria) aquí en vez de las métricas MAE/RMSE de `NB07` — ¿para qué tipo de problema está pensada cada una?

> **Para saber más**: [PyTorch: documentación del módulo `nn`](https://pytorch.org/docs/stable/nn.html) · [Tutorial básico de PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html).

> ***Como siempre: un modelo que "funciona sin errores" y un modelo que "realmente funciona bien" son barras muy distintas — hoy solo llegamos a la primera.***